## Instalando librerias

In [ ]:
!pip install clean-text
### Lematizacion
!python -m spacy download es_core_news_sm # no profundo
!python -m spacy download es_dep_news_trf # es profundo
!pip install spacy[transformers]

##Librerias

In [ ]:
import pandas as pd
# limpieza de texto
import html
import unicodedata
import re
from cleantext import clean
import string
import nltk
from nltk.corpus import stopwords
from nltk import word_tokenize
nltk.download('stopwords')
nltk.download('punkt')
from nltk.stem import SnowballStemmer
stemmer = SnowballStemmer('spanish')
import spacy
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

# Preprocesamiento de datos

## Importando datos

In [ ]:
path='/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/TwitterResults/EtiquetasTwitter/tweets_IM,_FP,_CH_19.09.2022.xlsx'

## Formacion politica

In [ ]:
fp=pd.read_excel(path,sheet_name=0)

In [ ]:
fp.info()

In [ ]:
fp=fp[['Twitter ID','Tweet']]

In [ ]:
fp=fp.assign(Categoria='FP',Etiqueta=1)

In [ ]:
fp.info()

In [ ]:
fp.head()

## Conciencia historica

In [ ]:
ch=pd.read_excel(path,sheet_name=1)

In [ ]:
ch.info()

In [ ]:
ch=ch[['Twitter ID','Tweet']]

In [ ]:
ch=ch.assign(Categoria='CH',Etiqueta=2)

In [ ]:
ch.info()

In [ ]:
ch.head()

## Imagen de la memoria

In [ ]:
im=pd.read_excel(path,sheet_name=2)

In [ ]:
im.info()

In [ ]:
im=im[['ID tweet','Tweet']]
im.columns=['Twitter ID','Tweet']

In [ ]:
im=im.assign(Categoria='IM',Etiqueta=3)

In [ ]:
im.info()

In [ ]:
im.head()

## Uniendo datos 

In [ ]:
data=pd.concat([fp,ch,im])

In [ ]:
data.info()

In [ ]:
data.head()

In [ ]:
data

## Limpieza de texto

In [ ]:
punct = set(string.punctuation) 
swsp=nltk.corpus.stopwords.words('spanish')
swen=nltk.corpus.stopwords.words('english')

In [ ]:
def clean_codigo_hex(texto):
  texto=html.unescape(texto)
  return texto

def clean_drop_unicode(texto):
  texto = unicodedata.normalize('NFD', texto)
  texto=texto.encode('ascii','ignore').decode('utf8')
  return texto

def clean_scape_sequence(texto):
  texto=re.sub('[\r\n\v\t\f\b]',' ',texto)
  return texto

def clean_drop_URLs(texto):
  text = re.sub('https?://\S+|www\.\S+', ' ', texto)
  return text

def clean_drop_URL(texto):
  text = re.sub('http?://\S+|www\.\S+', ' ', texto)
  return text

def clean_drop_emoticons(texto):
  texto=clean(texto, no_emoji=True,lower=False)
  return texto

def clean_drop_emails(texto):
  texto=clean(texto, no_emails=True, lower=False)
  return texto

def clean_drop_menciones(texto):
  texto = re.sub("@\S+", " ", texto)
  return texto

def clean_drop_numeros(texto):
  patron = r'[0-9]'
  texto = re.sub(patron, '', texto)
  return texto

def clean_drop_numeral(texto):
  texto = re.sub(r'#', ' ', texto)
  return texto

def split_palabras_adjuntas(texto):
  texto = " ".join([s for s in re.split("([A-Z][a-z]+[^A-Z]*)",texto) if s])
  return texto

def clean_drop_puntuacion(texto):
  sinpuntuacion="".join([ch for ch in texto if ch not in punct])
  return sinpuntuacion

def clean_drop_espaciosmas(texto):
  texto = re.sub('\s{2,}', " ", texto)
  return texto

def clean_drop_palabrascortas(texto):
  tokens = texto.split()
  clean_tokens = [t for t in tokens if len(t) > 1]
  clean_text = " ".join(clean_tokens)
  return clean_text

def clean_sin_mayusculas(texto):
  textonormalizado=texto.lower()
  return textonormalizado

def clean_drop_caracteres_repetidos(texto):
  texto = re.sub(r'(.)\1{3,}',r'\1', texto) # el numero 3 corresponde al numero maximo de repeticiones que se acepta
  return texto

def clean_drop_StopWords_Spanish(texto):
  if (str(texto) != 'nan'):
    tokens=word_tokenize(texto)
    tokensssw=[]
    for w in tokens:
      if w not in swsp:
        tokensssw.append(w)
    texto=" ".join(tokensssw)
  return texto

def clean_drop_StopWords_English(texto):
  if (str(texto) != 'nan'):
    tokens=word_tokenize(texto)
    tokensssw=[]
    for w in tokens:
      if w not in swen:
        tokensssw.append(w)
    texto=" ".join(tokensssw)
  return texto

In [ ]:
def cleaner_function(texto):
  texto_1=clean_codigo_hex(texto)
  texto_2=clean_drop_unicode(texto_1)
  texto_3=clean_scape_sequence(texto_2)
  texto_4=clean_drop_URLs(texto_3)
  texto_5=clean_drop_URL(texto_4)
  texto_6=clean_drop_emoticons(texto_5)
  texto_7=clean_drop_emails(texto_6)
  texto_8=clean_drop_menciones(texto_7)
  texto_9=clean_drop_numeros(texto_8)
  texto_10=clean_drop_numeral(texto_9)
  texto_11=split_palabras_adjuntas(texto_10)
  texto_12=clean_drop_puntuacion(texto_11)
  texto_13=clean_drop_espaciosmas(texto_12)
  texto_14=clean_drop_palabrascortas(texto_13)
  texto_15=clean_sin_mayusculas(texto_14)
  texto_16=clean_drop_caracteres_repetidos(texto_15)
  texto_17=clean_drop_StopWords_Spanish(texto_16)
  texto_limpio=clean_drop_StopWords_English(texto_17)
  return texto_limpio

In [ ]:
data=data.dropna()

In [ ]:
data.info()

In [ ]:
data['Tweet_Modificado']=data['Tweet'].apply(cleaner_function)

In [ ]:
data['Tweet']=data['Tweet'].apply(clean_scape_sequence)

In [ ]:
data

## Stemming 

In [ ]:
def stem(texto):
  doc_token=word_tokenize(texto)
  texto=" ".join([stemmer.stem(token) for token in doc_token])
  return texto

In [ ]:
data['Tweet_ModificadoStem']=data['Tweet_Modificado'].apply(stem)

In [ ]:
data.info()

In [ ]:
data.head()

## Lematizacion

In [ ]:
nlp = spacy.load("es_dep_news_trf")
nlp

In [ ]:
lemmatizer = nlp.get_pipe("lemmatizer")
print(lemmatizer.mode)  # 'rule'

In [ ]:
nlp.max_length=1100000

In [ ]:
def lematizarprofundo(texto):
  doc_spacy = nlp(texto)
  tokens_lema= " ".join([token.lemma_ for token in doc_spacy])
  return tokens_lema

In [ ]:
data['Tweet_ModificadoLematizadoProfundo']=data['Tweet_Modificado'].apply(lematizarprofundo)

In [ ]:
data.info()

In [ ]:
data.head()

## Exportar Dataframe a CSV

In [ ]:
data

In [ ]:
data.info()

In [ ]:
path='/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/TwitterResults/EtiquetasTwitter'

In [ ]:
data.to_excel(path+'/'+'RESULTS-EtiquetasTwitter.xlsx', index=False)

In [ ]:
data.to_csv(path+'/'+'RESULTS--EtiquetasTwitter.csv', index=False)

# Clasificador de Twitter

## Librerias

In [ ]:
import pandas as pd
import numpy as np
import sklearn
## Representacion de texto
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
## Modelos de clasificacion
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_val_score
## Evaluar el modelo
from sklearn import metrics
from sklearn.metrics import confusion_matrix
## Graficar
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

## Datos

In [ ]:
data=pd.read_csv('/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/TwitterResults/EtiquetasTwitter/RESULTS--EtiquetasTwitter.csv')

In [ ]:
data

In [ ]:
data=sklearn.utils.shuffle(data)

In [ ]:
data=data.dropna()

In [ ]:
data.info()

In [ ]:
ids = data["Tweet"]
data_rep=data[ids.isin(ids[ids.duplicated()])].sort_values("Tweet")

In [ ]:
data_rep

In [ ]:
path='/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/TwitterResults/EtiquetasTwitter'

In [ ]:
data_rep.to_excel(path+'/'+'RESULTS-EtiquetasTwitterRepetidas.xlsx', index=False)

In [ ]:
data_rep.to_csv(path+'/'+'RESULTS--EtiquetasTwitterRepetidas.csv', index=False)

In [ ]:
data

In [ ]:
conteo=data.Categoria.value_counts().rename_axis('Categoria').reset_index(name='Cantidad de etiquetas')
conteo

In [ ]:
sns.set_style("darkgrid")
fig, axes = plt.subplots(figsize=(7,7))
plt.title("Categorias de equipo Gumelab para Twitter")
grafica1=sns.barplot(x='Cantidad de etiquetas',y='Categoria',data=conteo, palette='crest')
plt.show()

## Representacion de texto

In [ ]:
vect = CountVectorizer()

In [ ]:
tfidf = TfidfVectorizer()

In [ ]:
text_train, text_test, y_train, y_test = train_test_split(data['Tweet_ModificadoLematizadoProfundo'], data['Etiqueta'], random_state = 0)

In [ ]:
vect = CountVectorizer().fit(text_train)
X_train_bag_of_words= vect.transform(text_train)
print("X_train_bag_of_words:\n{}".format(repr(X_train_bag_of_words)))

In [ ]:
print("Vocabulary size: {}".format(len(vect.vocabulary_)))

In [ ]:
tfidf = TfidfVectorizer().fit(text_train)
X_train_tfidf = tfidf.transform(text_train)
print("X_train_tfidf:\n{}".format(repr(X_train_tfidf)))

In [ ]:
print("Vocabulary size: {}".format(len(tfidf.vocabulary_)))

## Modelos de clasificacion

In [ ]:
models = [RandomForestClassifier(n_estimators=200, max_depth=3, random_state=0),LinearSVC(),MultinomialNB(),LogisticRegression(random_state=0)]

In [ ]:
CV = 5
cv_df = pd.DataFrame(index=range(CV * len(models)))
entries = []
for model in models:
  model_name = model.__class__.__name__
  accuracies = cross_val_score(model, X_train_bag_of_words, y_train, scoring='accuracy', cv=CV)
  for fold_idx, accuracy in enumerate(accuracies):
    entries.append((model_name, fold_idx, accuracy))
cv_df = pd.DataFrame(entries, columns=['Modelo', 'fold_idx', 'Accuracy'])

In [ ]:
cv_df.groupby('Modelo').Accuracy.mean()

In [ ]:
fig, axes = plt.subplots(figsize=(10,7))
sns.boxplot(x='Modelo', y='Accuracy', data=cv_df)
sns.stripplot(x='Modelo', y='Accuracy', data=cv_df,size=8, jitter=True, edgecolor="gray", linewidth=2)
plt.show()

In [ ]:
lr_bag_of_words = LogisticRegression(random_state=0).fit(X_train_bag_of_words, y_train)

In [ ]:
lr_bag_of_words.score(X_train_bag_of_words, y_train)

In [ ]:
print(lr_bag_of_words.predict(vect.transform([text_test.iloc[0]])))

In [ ]:
lr_tfidf = LogisticRegression(random_state=0).fit(X_train_tfidf, y_train)

In [ ]:
lr_tfidf.score(X_train_tfidf, y_train)

In [ ]:
print(lr_tfidf.predict(tfidf.transform([text_test.iloc[0]])))

In [ ]:
text_test

In [ ]:
len(text_test)

In [ ]:
data[data['Tweet_ModificadoLematizadoProfundo']==text_test.iloc[2]]

In [ ]:
data['Tweet_ModificadoLematizadoProfundo']

In [ ]:
len(data['Tweet_ModificadoLematizadoProfundo'])

In [ ]:
data['Tweet_ModificadoLematizadoProfundo'].iloc[0]

In [ ]:
print('Tweet: ', text_test.iloc[2])

In [ ]:
Prediccion_total=[]
Categoria_total=[]

In [ ]:
for i in range(len(data['Tweet_ModificadoLematizadoProfundo'])):
  prediccion=lr_bag_of_words.predict(vect.transform([data['Tweet_ModificadoLematizadoProfundo'].iloc[i]]))[0]
  Prediccion_total.append(prediccion)
  if prediccion==1:
    categoria="FP"
  elif prediccion==2:
    categoria="CH"
  elif prediccion==3:
    categoria="IM"
  Categoria_total.append(categoria)




In [ ]:
Prediccion_total

In [ ]:
Categoria_total

In [ ]:
data["Etiqueta_predicha"]=Prediccion_total

In [ ]:
data["Categoria_predicha"]=Categoria_total

In [ ]:
data

In [ ]:
Resultado=[]

In [ ]:
for i in range(len(data)):
  real=data["Etiqueta"].iloc[i]
  predicho=data["Etiqueta_predicha"].iloc[i]
  if real != predicho:
    resultado = False
  else:
    resultado = True
  Resultado.append(resultado) 

In [ ]:
Resultado

In [ ]:
data["Coincidencia_real_predicha"]=Resultado

In [ ]:
data

## Exportar Dataframe a CSV

In [ ]:
data

In [ ]:
data.info()

In [ ]:
path='/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/TwitterResults/EtiquetasTwitter'

In [ ]:
data.to_excel(path+'/'+'RESULTS-EtiquetasTwitterPredichas.xlsx', index=False)

In [ ]:
data.to_csv(path+'/'+'RESULTS--EtiquetasTwitterPredichas.csv', index=False)

In [ ]:
prediccion=lr_bag_of_words.predict(vect.transform([data['Tweet_ModificadoLematizadoProfundo'].iloc[0]]))[0]

In [ ]:
print('Etiqueta predicha: {} \nCorresponde a CH'.format(prediccion))

## Evaluando el modelo

In [ ]:
X_text_bag_of_words = vect.transform(text_test)
X_text_bag_of_words.shape

In [ ]:
y_pred = lr_bag_of_words.predict(X_text_bag_of_words)

In [ ]:
len(y_pred)

In [ ]:
y_pred_data=pd.DataFrame(y_pred)

In [ ]:
y_pred_data.value_counts()

In [ ]:
y_test

In [ ]:
len(y_test)

In [ ]:
y_test.value_counts()

## Matriz de confusion

In [ ]:
conf_mat = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(10,10))
sns.heatmap(conf_mat, annot=True, fmt='d',xticklabels=['FP','CH','IM'],yticklabels=['FP','CH','IM'])
plt.ylabel('Real')
plt.xlabel('Predicho')
plt.show()

## Informe de clasificacion para cada clase

In [ ]:
print(metrics.classification_report(y_test, y_pred,target_names=data['Categoria'].unique()))